# Stage 2B — CNN re-run (Q contamination fix)**Runtime → Change runtime type → T4 GPU → Save. Then Runtime → Run all.**---### Why this is being re-runDS1 holds only **8 Class Q beats**. Q was excluded from the evaluation metric butnot from the training objective. Two things followed:* **ADASYN** oversampled those 8 beats into roughly **12,800 synthetic ones** in  four of the five folds.* **Class weighting** gave Q a per-sample weight **126x** that of Class S, so 6  beats carried the same total gradient as 36,659 normal beats. Focal loss takes  the same weights as its alpha, so it was affected too.Every `class_weight`, `focal` and `adasyn` number from the previous run is void.The fix restricts both resampling and class weights to the evaluated classes(N, S, V, F). `smote` is also new, so the grid is 5 strategies instead of 4.---### Why there is a second re-runYour previous run was correct in itself, but it used a **different fold split**from the laptop half of the grid. `StratifiedGroupKFold` is deterministic withinone scikit-learn version and not across versions:| | scikit-learn | fold 0 validation beats ||---|---|---|| laptop | 1.9.1 | 10,467 || this Colab | 1.6.1 | 10,731 |Same data, same seed, different partition. Nothing errored — the LR/XGBoost rowsand the CNN rows simply described different splits of DS1, so comparing themmeant nothing.The fold assignment is now **frozen in `folds.json`**, which you upload alongwith the code. It was generated once on the laptop, so it matches the existingLR/XGBoost results exactly.**Do not run `make_folds.py`.** Your scikit-learn would regenerate a differentassignment and put us straight back here.### The code comes from the upload, not from DriveNothing is read from Drive except `ds1.npz`. Drive is used only for that and forbacking up results so a disconnect does not cost the run.**On a disconnect you will be asked to upload the six files again.** That takesa few seconds; completed folds are restored automatically.

## 1. Confirm a GPU was actually allocatedColab sometimes hands out a CPU runtime when GPUs are busy. This stops yourather than letting the run take eight times longer without saying so.

In [ ]:
import torchassert torch.cuda.is_available(), (    'No GPU allocated. Runtime -> Change runtime type -> T4 GPU, then Run all again.')print('GPU   :', torch.cuda.get_device_name(0))print('torch :', torch.__version__)

## 2. Mount Drive and locate `ds1.npz`A browser popup will ask you to authorise Drive access.`ds1.npz` is the dataset from Stage 1 — you already have it from the previousrun. The expected location is tried first; if it has moved, your Drive issearched for it.

In [ ]:
import globimport osfrom google.colab import drivedrive.mount('/content/drive')MYDRIVE = '/content/drive/MyDrive'WORK    = '/content/work'DS1 = f'{MYDRIVE}/ecg-paper/artifacts/ds1.npz'if not os.path.exists(DS1):    print('not at the expected path, searching your Drive...')    hits = glob.glob(f'{MYDRIVE}/**/ds1.npz', recursive=True)    assert hits, 'ds1.npz not found anywhere in your Drive -- ask Rishabh to send it'    DS1 = hits[0]print('ds1.npz :', DS1)!rm -rf {WORK}!mkdir -p {WORK}/src {WORK}/artifacts!cp "{DS1}" {WORK}/artifacts/ds1.npz%cd {WORK}!ls -la artifacts/

In [ ]:
# torch and sklearn ship with Colab; these two do not.!pip install -q xgboost imbalanced-learn

## 3. Upload the six filesA **Choose Files** button appears below. Select all six at once:```config.py   imbalance.py   models.py   run_grid.py   train_cnn.py   folds.json````folds.json` is the frozen fold assignment — the whole point of this re-run.Use the files Rishabh sent you this time. Do not substitute older copies fromDrive or from an earlier download; that is what went wrong on the first attempt.

In [ ]:
import osimport shutilfrom google.colab import filesNEEDED = {'config.py', 'imbalance.py', 'models.py', 'run_grid.py',          'train_cnn.py', 'folds.json'}print('Select all six files in one go:')print('   ', '  '.join(sorted(NEEDED)))print()uploaded = files.upload()os.makedirs(f'{WORK}/src', exist_ok=True)for name in uploaded:    shutil.move(name, f'{WORK}/src/{name}')got = {f for f in os.listdir(f'{WORK}/src')       if f.endswith(('.py', '.json'))}missing = NEEDED - gotextra = got - NEEDEDprint()for f in sorted(got):    print(f'  {f:18} {os.path.getsize(f"{WORK}/src/{f}"):>7,} bytes')if extra:    print('\nunexpected files (duplicates from a second upload?):', sorted(extra))assert not missing, (f'missing: {sorted(missing)} -- run this cell again '                     'and select all six')print('\nall six uploaded')

## 4. Verify the code is the fixed version**If anything fails here, stop and tell Rishabh** — do not try to work around it.A failure means the uploaded files are not the ones from the fix, and the runwould spend an hour producing the same wrong numbers again.

In [ ]:
import sysimport numpy as npsys.path.insert(0, 'src')import config as Cfrom imbalance import class_weights, STRATEGIESy = np.load('artifacts/ds1.npz')['y']counts = np.bincount(y, minlength=5)w = class_weights(y)print('beats      :', f'{len(y):,}', '  (expect 50,977)')print('classes    :', dict(zip(C.CLASS_NAMES, counts.tolist())))print('eval on    :', C.EVAL_CLASSES, '  (Q excluded)')print('strategies :', STRATEGIES)print()print('class weights:', {c: round(float(v), 5) for c, v in zip(C.CLASS_NAMES, w)})print('Q weight     :', float(w[4]), '  <-- MUST be exactly 0.0')assert len(y) == 50977, 'wrong ds1.npz'assert 'smote' in STRATEGIES, "OLD imbalance.py -- 'smote' missing. Tell Rishabh."assert float(w[4]) == 0.0,    'OLD imbalance.py -- Q still weighted. Tell Rishabh.'# The frozen fold assignment. Fold 0 must hold 10,467 validation beats --# that is the laptop's split, the one the LR/XGBoost results already use.import jsonwith open('src/folds.json') as fh:    spec = json.load(fh)pf = {str(k): int(v) for k, v in spec['patient_fold'].items()}pid = np.load('artifacts/ds1.npz')['pid']assign = np.array([pf[str(p)] for p in pid])sizes = [int((assign == f).sum()) for f in range(spec['n_folds'])]print()print('folds.json  :', spec['n_patients'], 'patients,', spec['n_folds'], 'folds')print('made with   : sklearn', spec['generated_with']['sklearn'])print('fold sizes  :', sizes, '  <-- fold 0 must be 10,467')assert spec['n_beats'] == len(y), 'folds.json does not match this ds1.npz'assert sizes[0] == 10467, (    f'fold 0 has {sizes[0]} beats, expected 10,467 -- wrong folds.json')print('\nOK -- fixed code and frozen folds confirmed')

## 5. Smoke test — about 2 minutes, do not skipA tiny subset with 5 epochs. The numbers it prints are meaningless; the point isto find out whether anything crashes before committing to an hour-long run.This exercises the CNN's SMOTE path, which has never run before.

In [ ]:
!python src/run_grid.py --models cnn --strategies smote --rr 1 --subset 5000 --epochs 5

In [ ]:
# Smoke-test predictions use the same filenames as real ones, so they are# deleted immediately. run_grid.py also validates cached folds against the# current split, but two guards are better than one here.!rm -rf {WORK}/artifacts/preds {WORK}/artifacts/grid_summary.jsonprint('smoke-test output cleared')

## 6. First run or resume?This decides automatically, using a marker file kept on your Drive:* **First run** — any CNN predictions already on Drive are from before the fix,  so they are removed and the run starts clean.* **Resume after a disconnect** — completed folds are copied back and the run  picks up where it stopped.Running the whole notebook again after a disconnect is therefore always safe.

In [ ]:
import globimport osimport shutilPREDS_DRIVE = f'{MYDRIVE}/ecg-paper/artifacts/preds'PREDS_WORK  = f'{WORK}/artifacts/preds'MARKER      = f'{PREDS_DRIVE}/.qfix_v2'os.makedirs(PREDS_DRIVE, exist_ok=True)os.makedirs(PREDS_WORK, exist_ok=True)if os.path.exists(MARKER):    done = sorted(glob.glob(f'{PREDS_DRIVE}/cnn_*'))    for f in done:        shutil.copy2(f, PREDS_WORK)    print(f'RESUMING -- restored {len(done)} completed CNN folds')    print(f'           {50 - len(done)} still to compute')else:    stale = glob.glob(f'{PREDS_DRIVE}/cnn_*')    for f in stale:        os.remove(f)    open(MARKER, 'w').close()    print(f'FIRST RUN -- removed {len(stale)} pre-fix CNN folds from Drive')    print('           all 50 folds will be computed')

## 7. The real run — 50 to 60 minutes5 strategies x 2 RR settings x 5 folds = **50 fold-runs**.Each fold is written to disk the moment it finishes, so nothing is lost on adisconnect. One line prints per configuration as it completes.**Keep this tab open and click on the page every so often** — Colab drops idlesessions after about 90 minutes.

In [ ]:
!python src/run_grid.py --models cnn

## 8. Back up to DriveColab's local disk is wiped on disconnect; Drive is what survives, and thesefiles are the resume points.**Run this cell again every 20 to 25 minutes while the run above is going** —click into it and press Ctrl+Enter. It is safe to run at any time.

In [ ]:
!mkdir -p "{PREDS_DRIVE}"!cp -r {WORK}/artifacts/preds/* "{PREDS_DRIVE}/" 2>/dev/nulln = !ls {WORK}/artifacts/preds | grep ^cnn | wc -lprint(f'{n[0].strip()} CNN folds backed up   (50 when complete)')

## 9. ResultsThe table, an automatic check of the RR ablation, and the three numbers toreport back.

In [ ]:
import jsonwith open('artifacts/grid_summary.json') as f:    rows = json.load(f)by = {(r['strategy'], r['use_rr']): r for r in rows}print(f"{'strategy':14} {'rr':>2}   {'macro-F1':>17}   {'S':>6} {'V':>6} {'F':>6}")print('-' * 62)for r in rows:    pc = r['per_class_f1']    print(f"{r['strategy']:14} {int(r['use_rr']):>2}   "          f"{r['macro_f1_mean']:.4f} +/- {r['macro_f1_std']:.4f}   "          f"{pc['S']:6.3f} {pc['V']:6.3f} {pc['F']:6.3f}")def val(r, what):    return r['macro_f1_mean'] if what == 'macro-F1' else r['per_class_f1']['S']print('\n' + '=' * 62)print('RR ABLATION  --  rr=1 must beat rr=0 in every row')print('=' * 62)ok = tot = 0for s in sorted({r['strategy'] for r in rows}):    hi_r, lo_r = by.get((s, True)), by.get((s, False))    if not hi_r or not lo_r:        continue    for what in ('macro-F1', 'S'):        hi, lo = val(hi_r, what), val(lo_r, what)        tot += 1        ok += hi > lo        flag = 'pass' if hi > lo else '*** FAIL ***'        print(f'  {s:14} {what:9} {hi:.4f}  vs  {lo:.4f}   {flag}')print(f'\n  {ok}/{tot} comparisons passed', '' if ok == tot else '  <-- REPORT THIS')print('\n' + '=' * 62)print('THE THREE NUMBERS TO REPORT  (rr=1, S-class F1)')print('=' * 62)for s in ('class_weight', 'adasyn', 'smote'):    r = by.get((s, True))    if r:        print(f'  cnn {s:14} S = {r["per_class_f1"]["S"]:.3f}')base = by.get(('none', True))if base:    print(f'  (baseline  none         S = {base["per_class_f1"]["S"]:.3f})')

In [ ]:
# Did resampling actually happen? Each prediction records what the sampler did.import globimport jsonimport numpy as npnotes = sorted({json.loads(np.load(p)['meta'].item())['resample']                for p in glob.glob('artifacts/preds/cnn_*')})print('resample notes:', notes)bad = [n for n in notes if 'failed' in n or 'skipped_no_eligible' in n]print('\nPROBLEM -- report these:', bad) if bad else print('\nno sampler failures')

## 10. Package the predictions and download themThis writes a single zip and downloads it to your computer. Send that file toRishabh — it is the deliverable.

In [ ]:
import globimport osimport zipfileOUT = '/content/partB_preds.zip'folds = sorted(glob.glob(f'{WORK}/artifacts/preds/cnn_*'))with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:    for f in folds:        z.write(f, 'preds/' + os.path.basename(f))print(f'{len(folds)} CNN folds packed   (expect 50)')print(f'{os.path.getsize(OUT) / 1e6:.1f} MB -> {OUT}')if len(folds) != 50:    print('\nWARNING: not 50 folds. Finish the run before sending this.')

In [ ]:
from google.colab import filesfiles.download('/content/partB_preds.zip')

## 11. What to send back1. `partB_preds.zip` — downloaded by the cell above.2. The full output of **cell 9** — the table, the ablation result, the three   numbers.### Stop and report instead of continuing if you see| What | What it means ||---|---|| any assertion failure in cell 3 or 4 | wrong files uploaded; the run would be wasted || `folds.json is missing` | you did not upload it — re-run cell 3 || macro-F1 above 0.85 | almost certainly a data leak || `focal` identical to `none` | the loss is not being applied || any FAIL in the ablation check | the central result of the paper did not reproduce || anything in the sampler-failure list | ADASYN or SMOTE silently did nothing |### If Colab disconnectsRun all the cells again from the top. You will have to re-upload the six filesin cell 3 — that is the only manual step. Cell 6 restores completed foldsand cell 7 skips them, so nothing computed so far is lost, as long as cell 8 wasrun recently.